# Project FORESIGHT — Phase 2B: Baseline Forecasting Framework

**Objective**: Establish a leakage-safe temporal evaluation protocol, aggregate daily demand to weekly grain, evaluate 5 classical baselines across 12 rolling origins and 8-week horizons, and define the performance benchmark for ML models.

**Strict Boundary**: Classical baselines ONLY. No ML models, no hyperparameter tuning, no predictive feature engineering.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add project root to path
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CFG, PATHS
from src.baseline import (
    load_analysis_ready_data,
    aggregate_weekly_demand,
    build_rolling_origins,
    generate_baseline_forecasts,
    compute_overall_metrics,
    compute_macro_metrics,
    compute_metrics_by_horizon,
    compute_metrics_by_category,
    compute_metrics_by_sku,
    verify_data_integrity,
)

print(f'FORESIGHT Baseline Framework Initialized. Random seed: {CFG.random_seed}')


## 1. Load Analysis-Ready Data & Aggregate to Weekly Grain

Following Phase 2A findings, daily demand is aggregated to weekly grain (ISO Mon-Sun, W-MON anchor). This raises the signal-to-noise ratio (SNR) from 1.65 to 3.20 while eliminating intra-week day-of-week noise.


In [2]:
daily_df = load_analysis_ready_data()
weekly_df = aggregate_weekly_demand(daily_df)
print(f'Daily Shape: {daily_df.shape} (36,550 rows, 50 SKUs x 731 dates)')
print(f'Weekly Shape: {weekly_df.shape} (50 SKUs x 106 weeks)')
print(f'Daily Total Units: {daily_df["Units_Sold"].sum():,}')
print(f'Weekly Total Units: {weekly_df["Units_Sold"].sum():,}')
assert daily_df['Units_Sold'].sum() == weekly_df['Units_Sold'].sum(), 'Total volume reconciliation failed!'
print('Reconciliation check: PASSED (Volume conservation verified)')


## 2. Rolling-Origin Backtesting Configuration

Walk-forward rolling origins ensure zero future information leakage:
- Minimum training: 52 weeks (1 full year)
- Forecast horizon: 8 weeks (h=1..8)
- 12 rolling folds stepping ~3-4 weeks across the validation period
- Final holdout: Weeks 97-104 preserved untouched


In [3]:
origins = build_rolling_origins(weekly_df, n_origins=12, horizon=8, min_train_weeks=52)
print(f'Generated {len(origins)} rolling origins:')
for i, o in enumerate(origins, 1):
    print(f'  Fold {i:02d}: Origin Date = {o.strftime("%Y-%m-%d")}')


## 3. Generate Baseline Forecasts Across 5 Methods

Models evaluated:
1. **Naive**: y_hat(t+h) = y(t)
2. **Seasonal Naive**: y_hat(t+h) = y(t+h-52)
3. **MA4**: y_hat(t+h) = mean(y[t-3:t])
4. **MA8**: y_hat(t+h) = mean(y[t-7:t])
5. **SES (alpha=0.3)**: Exponential smoothing on historical observations


In [4]:
forecasts_df = generate_baseline_forecasts(weekly_df, origins, horizon=8)
print(f'Generated {len(forecasts_df):,} forecast records (12 origins x 50 SKUs x 8 horizons x 5 models).')
print(forecasts_df.head())


## 4. Overall & Macro Model Performance

Primary metric: **WAPE** (Weighted Absolute Percentage Error = sum|y-y_hat|/sum|y| * 100%).
Secondary metrics: **MAE**, **RMSE**.


In [5]:
summary_df = compute_overall_metrics(forecasts_df)
macro_df = compute_macro_metrics(forecasts_df)
print('=== Overall (Micro / Volume-Weighted) Performance ===')
print(summary_df.to_string(index=False))
print('\n=== Macro (Unweighted Per-SKU Average) Performance ===')
print(macro_df.to_string(index=False))


## 5. Performance by Forecast Horizon (h=1..8)

Evaluates error growth as the lead time extends from 1 to 8 weeks.


In [6]:
horizon_df = compute_metrics_by_horizon(forecasts_df)
pivot_wape = horizon_df.pivot(index='horizon', columns='model', values='WAPE').round(2)
print('=== WAPE (%) by Horizon ===')
print(pivot_wape)


## 6. Category-Level Performance


In [7]:
cat_df = compute_metrics_by_category(forecasts_df)
cat_pivot = cat_df.pivot(index='Category', columns='model', values='WAPE').round(2)
print('=== WAPE (%) by Category ===')
print(cat_pivot)


## 7. Baseline Visualizations Overview


In [8]:
from IPython.display import Image, display
plot_dir = PATHS.artifacts_dir / 'baseline' / 'plots'
plots = sorted(list(plot_dir.glob('*.png')))
print(f'Found {len(plots)} visualization plots in {plot_dir}:')
for p in plots:
    print(f'  - {p.name}')


## 8. Data Integrity Verification

Verify that analysis_ready.parquet and raw source CSV files were completely unmodified.


In [9]:
status = verify_data_integrity()
print(f'Data Integrity Status: {"PASSED [OK]" if status else "FAILED"}')
